# Tutorial 7: Train NicheTrans on 10x Xenium data

In [4]:
import os, time, datetime, warnings

import torch
import torch.nn as nn
from torch.optim import lr_scheduler

from model.nicheTrans_img import *
from datasets.data_manager_breast_cancer import Breast_cancer

from utils.utils import *
from utils.utils_training_breast_cancer import train, test
from utils.utils_dataloader import *

warnings.filterwarnings("ignore")

### Initialize the args and fix seeds

In [5]:
%run ./args/args_breast_cancer.py
args = args

set_seed(args.seed)
os.environ['CUDA_VISIBLE_DEVICES'] = args.gpu_devices

print("==========\nArgs:{}\n==========".format(args))

Args:Namespace(noise_rate=0.2, dropout_rate=0.2, workers=4, adata_path='/home1/shezixi/data/in_silico_central_dogma_challenge/unprocessed/2023_nc_10x_breast_cancer/HBC_rep1_cell_nucleus_3channel_strength_mean.h5ad', coordinate_path='/home1/shezixi/data/in_silico_central_dogma_challenge/unprocessed/2023_nc_10x_breast_cancer/cells.csv.gz', ct_path='/home1/shezixi/data/in_silico_central_dogma_challenge/unprocessed/2023_nc_10x_breast_cancer/Cell_Barcode_Type_Matrices.xlsx', max_epoch=40, stepsize=20, train_batch=32, test_batch=32, optimizer='adam', lr=0.0003, gamma=0.1, weight_decay=0.0005, seed=1, eval_step=1, gpu_devices='0')


### Initialize dataloaders and NicheTrans

In [6]:
# create the dataloaders
dataset = Breast_cancer(adata_path=args.adata_path, coordinate_path=args.coordinate_path, ct_path=args.ct_path)
trainloader, testloader = breast_cancer_dataloader(args, dataset)

# create the model
source_dimension, target_dimension = dataset.rna_length, dataset.protein_length
model = NicheTrans(source_length=source_dimension, target_length=target_dimension, noise_rate=args.noise_rate, dropout_rate=args.dropout_rate)
model = nn.DataParallel(model).cuda()

------Calculating spatial graph...
The graph contains 1185564 edges, 98797 cells.
12.0000 neighbors per cell on average.
------Calculating spatial graph...
The graph contains 827796 edges, 68983 cells.
12.0000 neighbors per cell on average.
=> AD Mouse loaded
Dataset statistics:
  ------------------------------
  subset   | # num | 
  ------------------------------
  train    |  98797 spots, 98659 positive CD20, 84043 positive HER2 
  test     |  68983 spots, 67600 positive CD20, 36904 positive HER2 
  ------------------------------


### Load bio-prior

In [7]:
from prior_AddOn.gene_embedding_loader import load_static_gene_prior
priors = load_static_gene_prior(
    source_panel=dataset.source_panel,
    models=("scgpt",),
    species="human",  
    root="prior_AddOn/gene_embeddings",
)


In [8]:
for row in priors["scgpt"]["mapping_table"]:
    if row["status"] != "mapped":
        print(row["input_gene"], row["status"], row["reason"])

POLR2J3 missing_embedding human_symbol_not_in_scgpt_vocab
QARS missing_embedding human_symbol_not_in_scgpt_vocab


In [12]:
priors["scgpt"]["coverage"]

{'model': 'scgpt',
 'species': 'human',
 'n_features': 313,
 'n_found': 311,
 'coverage': 0.9936102236421726,
 'by_status': {'mapped': 311, 'missing_embedding': 2}}